In [1]:
!pip install -q pdfplumber sentence-transformers faiss-cpu gradio anthropic

In [2]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/rag_buku_kia'
import os
os.makedirs(PROJECT_DIR, exist_ok=True)
print('Project dir:', PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project dir: /content/drive/MyDrive/rag_buku_kia


In [3]:
import pdfplumber
import re

def clean_text(text):
    text = re.sub(r'\n\d{1,3}\n', '\n', text)
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def extract_physical_pages(pdf_path):
    physical_pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            width, height = page.width, page.height
            halves = {
                'left': page.crop((0, 0, width / 2, height)),
                'right': page.crop((width / 2, 0, width, height)),
            }
            for side, half_page in halves.items():
                raw_text = half_page.extract_text() or ''
                tables = half_page.extract_tables()
                physical_pages.append({
                    'pdf_page': i + 1,
                    'side': side,
                    'text': clean_text(raw_text),
                    'tables': tables,
                })
    return physical_pages

physical_pages = extract_physical_pages('/content/buku_kia.pdf')
print(f'Total halaman fisik: {len(physical_pages)}')
print(physical_pages[10]['text'][:400])

Total halaman fisik: 160
KEHAMILAN - TRIMESTER 1
Tanda Bahaya Pada Trimester 1
Harus diketahui ibu dan keluarga.
Jika mengalami tanda bahaya pada masa kehamilan, segera bawa
ibu hamil periksa ke Puskesmas/Rumah Sakit.
Demam Tinggi. Nyeri perut hebat.
Sakit saat kencing atau keluar
keputihan atau gatal di daerah
Mual dan muntah hebat. Perdarahan. kemaluan.
Masalah Lain Pada Kehamilan
Demam, menggigil dan berkeringat. Bila 


In [4]:
def get_section_title(text):
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    return lines[0] if lines else 'UNKNOWN'

def format_table_as_text(table):
    rows_text = []
    for row in table:
        cells = [c.strip() if c else '' for c in row]
        rows_text.append(' | '.join(cells))
    return '\n'.join(rows_text)

for p in physical_pages:
    p['section_title'] = get_section_title(p['text'])
    if p['tables']:
        p['table_text'] = '\n\n'.join(format_table_as_text(t) for t in p['tables'])
    else:
        p['table_text'] = ''

from collections import Counter
title_counts = Counter(p['section_title'] for p in physical_pages)
for title, count in title_counts.most_common(20):
    print(f'{count:3d}  {title}')

 19  PENGUKURAN & PENCATATAN OLEH TENAGA KESEHATAN
 11  (cid:39)(cid:2)(cid:8)(cid:46)(cid:19)(cid:1)(cid:19)(cid:12)(cid:4)(cid:8)(cid:9)(cid:15)(cid:9)(cid:39)(cid:2)(cid:8)(cid:28)(cid:4)(cid:11)(cid:4)(cid:11)(cid:4)(cid:8)(cid:9)(cid:45)(cid:7)(cid:2)(cid:3)(cid:9)(cid:11)(cid:2)(cid:8)(cid:4)(cid:46)(cid:4)(cid:9)(cid:1)(cid:2)(cid:13)(cid:2)(cid:3)(cid:4)(cid:11)(cid:4)(cid:8)
  9  INFORMASI UMUM
  8  0 - 6 BULAN
  8  (cid:16)(cid:9)(cid:10)(cid:9)(cid:21)(cid:9)(cid:11)(cid:4)(cid:3)(cid:19)(cid:8)
  5  KEHAMILAN - TRIMESTER 1
  5  (cid:5)(cid:2)(cid:7)(cid:4)(cid:3)(cid:6)(cid:12)(cid:1)(cid:4)(cid:8)
  4  SETELAH MELAHIRKAN
  4  (cid:20)(cid:9)(cid:10)(cid:9)(cid:21)(cid:9)(cid:22)(cid:19)(cid:7)(cid:4)(cid:8)
  4  6 - 12 BULAN
  4  12 - 24 BULAN
  3  KEHAMILAN - TRIMESTER 2
  3  (cid:21)(cid:9)(cid:10)(cid:9)(cid:14)(cid:16)(cid:9)(cid:22)(cid:19)(cid:7)(cid:4)(cid:8)
  3  (cid:10)(cid:11)(cid:14)(cid:11)(cid:8)(cid:7)(cid:3)(cid:7)(cid:44)(cid:4)(cid:11)(cid:12)(cid:11)(cid

In [5]:
EXCLUDE_TITLES = {'UNKNOWN'}  # tambahkan title lain setelah dicek manual dari output di atas

content_pages = [p for p in physical_pages if p['section_title'] not in EXCLUDE_TITLES]
print(f'Halaman dipakai: {len(content_pages)} dari {len(physical_pages)}')

Halaman dipakai: 158 dari 160


In [6]:
TAHAP_KEYWORDS = {
    'hamil': ['hamil', 'kehamilan', 'trimester', 'kandungan'],
    'bersalin': ['persalinan', 'melahirkan', 'bersalin'],
    'menyusui': ['menyusui', 'asi', 'laktasi'],
    'bayi': ['bayi baru lahir', 'perawatan bayi', 'imunisasi', 'balita'],
}

DANGER_SECTION_HINTS = ['tanda bahaya', 'tidak boleh', 'segera bawa', 'segera ke', 'gawat darurat']
DANGER_TEXT_HINTS = ['perdarahan', 'pendarahan', 'kejang', 'demam tinggi', 'rujuk', 'segera bawa']

def label_page(page):
    combined_lower = (page['section_title'] + ' ' + page['text']).lower()
    tahap = 'umum'
    for key, kws in TAHAP_KEYWORDS.items():
        if any(kw in combined_lower for kw in kws):
            tahap = key
            break
    is_danger_sign = (
        any(kw in page['section_title'].lower() for kw in DANGER_SECTION_HINTS)
        or any(kw in combined_lower for kw in DANGER_TEXT_HINTS)
    )
    return {**page, 'tahap': tahap, 'is_danger_sign': is_danger_sign}

labeled_pages = [label_page(p) for p in content_pages]

all_chunks = []
for p in labeled_pages:
    full_text = p['text']
    if p['table_text']:
        full_text += '\n\n[Tabel]\n' + p['table_text']
    if not full_text.strip():
        continue
    all_chunks.append({
        'chunk_id': f"page{p['pdf_page']}_{p['side']}",
        'text': full_text,
        'section_title': p['section_title'],
        'tahap': p['tahap'],
        'is_danger_sign': p['is_danger_sign'],
        'pdf_page': p['pdf_page'],
    })

print(f'Total chunk: {len(all_chunks)}')

Total chunk: 158


In [16]:
import re

CAPTION_PATTERN = re.compile(r'^[A-Z][a-zA-Z ,\-]{2,40}\.$')

def split_into_items(text):
    """
    Coba pecah teks jadi sub-blok berdasarkan baris yang mirip caption/label pendek.
    Return list of (label, body). Kalau tidak ada caption terdeteksi (< 2), return
    seluruh teks sebagai satu blok dengan label None.
    """
    lines = text.split('\n')
    caption_indices = [i for i, l in enumerate(lines) if CAPTION_PATTERN.match(l.strip())]

    if len(caption_indices) < 2:
        return [(None, text)]

    items = []
    for idx, start in enumerate(caption_indices):
        label = lines[start].strip()
        end = caption_indices[idx + 1] if idx + 1 < len(caption_indices) else len(lines)
        body_lines = lines[start:end]
        items.append((label, '\n'.join(body_lines).strip()))
    return items

sub_chunks = []
for c in all_chunks:
    items = split_into_items(c['text'])
    if len(items) == 1 and items[0][0] is None:
        sub_chunks.append(c)
    else:
        for idx, (label, body) in enumerate(items):
            sub_chunks.append({
                **c,
                'chunk_id': f"{c['chunk_id']}_item{idx}",
                'text': body,
                'item_label': label,
            })

print(f'Total chunk sebelum sub-chunking: {len(all_chunks)}')
print(f'Total chunk sesudah sub-chunking: {len(sub_chunks)}')

all_chunks = sub_chunks  # pakai hasil sub-chunking untuk tahap selanjutnya

Total chunk sebelum sub-chunking: 158
Total chunk sesudah sub-chunking: 158


In [19]:
import json

chunks_path = f'{PROJECT_DIR}/chunks.json'
with open(chunks_path, 'w', encoding='utf-8') as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=2)
print('Chunks tersimpan di:', chunks_path)

Chunks tersimpan di: /content/drive/MyDrive/rag_buku_kia/chunks.json


In [20]:
import sys
!{sys.executable} -m pip install Pillow==9.5.0

from sentence_transformers import SentenceTransformer
import numpy as np

EMBED_MODEL_NAME = 'intfloat/multilingual-e5-base'
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

chunk_texts = ['passage: ' + c['text'] for c in all_chunks]
chunk_embeddings = embed_model.encode(chunk_texts, show_progress_bar=True, convert_to_numpy=True)

print('Shape embedding:', chunk_embeddings.shape)
np.save(f'{PROJECT_DIR}/chunk_embeddings.npy', chunk_embeddings)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Shape embedding: (158, 768)


In [21]:
import faiss

dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)

faiss.normalize_L2(chunk_embeddings)
index.add(chunk_embeddings)

faiss.write_index(index, f'{PROJECT_DIR}/faiss_index.bin')
print('Jumlah vector di index:', index.ntotal)

Jumlah vector di index: 158


In [22]:
def keyword_boost(query, chunk_text, chunk_label=None):
    query_words = set(w.lower() for w in query.split() if len(w) > 3)
    text_lower = (chunk_text + ' ' + (chunk_label or '')).lower()
    matches = sum(1 for w in query_words if w in text_lower)
    return matches * 0.05  # boost kecil per kata kunci yang cocok persis

def retrieve(query, top_k=5, initial_k=15):
    query_vec = embed_model.encode(['query: ' + query], convert_to_numpy=True)
    faiss.normalize_L2(query_vec)
    scores, indices = index.search(query_vec, initial_k)

    candidates = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        chunk = all_chunks[idx]
        boost = keyword_boost(query, chunk['text'], chunk.get('item_label'))
        candidates.append({**chunk, 'score': float(score) + boost})

    candidates.sort(key=lambda c: c['score'], reverse=True)
    return candidates[:top_k]

test_results = retrieve('apa tanda bahaya di trimester 1?')
for r in test_results:
    print(f"[{r['score']:.3f}] p.{r['pdf_page']} {r['section_title']} (danger={r['is_danger_sign']})")
    print(r['text'][:150], '...\n')

[1.061] p.6 KEHAMILAN - TRIMESTER 1 (danger=True)
KEHAMILAN - TRIMESTER 1
Tanda Bahaya Pada Trimester 1
Harus diketahui ibu dan keluarga.
Jika mengalami tanda bahaya pada masa kehamilan, segera bawa
i ...

[1.056] p.10 KEHAMILAN - TRIMESTER 2 (danger=True)
KEHAMILAN - TRIMESTER 2
Tanda Bahaya Pada Trimester 2
Jika mengalami tanda bahaya pada masa kehamilan, segera bawa
ibu hamil periksa ke Puskesmas/Ruma ...

[1.000] p.5 KEHAMILAN - TRIMESTER 1 (danger=False)
KEHAMILAN - TRIMESTER 1
Usia Kehamilan 1-3 Bulan (Trimester 1)
Masa Penting Pembentukan Bagian Tubuh Bayi
beras, hingga 10 cm dan berat sekitar 28
gra ...

[1.000] p.11 KEHAMILAN - TRIMESTER 3 (danger=False)
KEHAMILAN - TRIMESTER 3
Usia Kehamilan 7-9 Bulan (Trimester 3)
Persiapan Menyambut Kehadiran Si Kecil
YANG HARUS DILAKUKAN
• Periksa kehamilan paling  ...

[0.996] p.10 KEHAMILAN - TRIMESTER 2 (danger=False)
KEHAMILAN - TRIMESTER 2
Usia Kehamilan 4-6 Bulan (Trimester 2)
Saatnya Mulai Merencanakan Kelahiran
YANG HARUS DILAKUKAN

In [11]:
!pip install -U bitsandbytes>=0.46.1

In [31]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# Default 3B — lebih aman untuk GPU T4 16GB. Kalau VRAM kamu lega dan mau kualitas lebih
# tinggi, boleh coba ganti ke 'Qwen/Qwen2.5-7B-Instruct', tapi pastikan cek Step di bawah
# yang memverifikasi quantization benar-benar ke-apply sebelum lanjut generate.
GEN_MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)

gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
gen_model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
)
print('Model generation siap:', GEN_MODEL_NAME)

# Verifikasi quantization benar-benar ke-apply (penting! kalau ini gagal diam-diam,
# model ke-load full precision dan gampang OOM di T4)
is_quantized = getattr(gen_model, 'is_loaded_in_4bit', False)
print('Ter-load dalam 4-bit:', is_quantized)
if torch.cuda.is_available():
    print('VRAM terpakai setelah load model:', round(torch.cuda.memory_allocated() / 1e9, 2), 'GB')
if not is_quantized:
    print('PERINGATAN: model TIDAK ter-quantize. Restart runtime dan cek versi bitsandbytes/accelerate.')

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Model generation siap: Qwen/Qwen2.5-3B-Instruct
Ter-load dalam 4-bit: True
VRAM terpakai setelah load model: 8.43 GB


In [32]:
SYSTEM_PROMPT = """Kamu adalah asisten informasi kesehatan ibu hamil, bersalin, dan menyusui
berdasarkan Buku KIA (Kemenkes RI).

ATURAN KETAT:
1. Jawab HANYA berdasarkan konteks yang diberikan. Jangan menambahkan informasi dari luar konteks.
2. Jika konteks tidak cukup untuk menjawab, katakan dengan jujur bahwa informasi tidak tersedia
   dan sarankan konsultasi ke bidan/dokter.
3. Ini BUKAN pengganti konsultasi medis. Selalu sertakan pengingat ini bila relevan.
4. Jika ada tanda bahaya dalam pertanyaan atau konteks, WAJIB sarankan segera ke fasilitas
   kesehatan terdekat sebelum informasi lainnya.
"""

MAX_CHARS_PER_CHUNK = 600  # batasi tiap chunk di prompt supaya konteks gak kepanjangan

def generate_answer(query, retrieved_chunks, max_new_tokens=800):
    context = '\n\n'.join(
        f"[{c['section_title']} - p.{c['pdf_page']}] {c['text'][:MAX_CHARS_PER_CHUNK]}"
        for c in retrieved_chunks
    )
    has_danger = any(c['is_danger_sign'] for c in retrieved_chunks)
    danger_note = (
        "\n\nPERHATIAN: Konteks ini mengandung informasi tanda bahaya. "
        "Pastikan jawabanmu menyertakan arahan untuk segera ke fasilitas kesehatan."
        if has_danger else ''
    )

    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT + danger_note},
        {'role': 'user', 'content': f"Konteks:\n{context}\n\nPertanyaan: {query}"},
    ]

    prompt = gen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = gen_tokenizer(prompt, return_tensors='pt').to(gen_model.device)

    with torch.no_grad():
        output_ids = gen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=gen_tokenizer.eos_token_id,
        )

    generated = output_ids[0][inputs['input_ids'].shape[1]:]
    answer = gen_tokenizer.decode(generated, skip_special_tokens=True).strip()

    # Bersihkan cache GPU setelah tiap generation supaya gak menumpuk antar pemanggilan
    del inputs, output_ids, generated
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return answer

answer = generate_answer('apa tanda bahaya di trimester 1?', test_results)
print(answer)

Tanda bahaya di trimester 1 antara lain:

1. Demam tinggi
2. Nyeri perut hebat
3. Sakit saat kencing atau keluar keputihan/gatal di daerah kemaluan
4. Mual dan muntah hebat
5. Perdarahan
6. Batuk lama (lebih dari 2 minggu)

Jika mengalami salah satu tanda bahaya tersebut, segera bawa ibu hamil untuk periksa ke Puskesmas/Rumah Sakit.


In [33]:
test_questions = [
    # 'apa tanda bahaya di trimester 1?',
    # 'bagaimana cara memerah dan menyimpan ASI?',
    # 'berapa porsi makan ibu menyusui per hari?',
    # 'aktivitas fisik apa yang tidak boleh dilakukan ibu hamil?',
    # 'berapa lama ASI perah bisa disimpan di kulkas?',
    'cara membuat bubur sup daging kacang merah',  # kasus yang tadi tertukar dengan demam
]

for q in test_questions:
    print('=' * 60)
    print('Q:', q)
    results = retrieve(q, top_k=3)
    ans = generate_answer(q, results)
    print('A:', ans)
    print()

Q: cara membuat bubur sup daging kacang merah
A: Berdasarkan informasi yang disampaikan, tidak ada detail spesifik tentang cara membuat bubur sup daging kacang merah di buku tersebut. Untuk informasi lebih lengkap, saya sarankan Anda berkonsultasi dengan bidan atau dokter. Namun, berikut adalah contoh dasar cara membuat bubur sup daging kacang merah:

### Cara Membuat Bubur Sup Daging Kacang Merah

#### Bahan-bahan:
- 200 gram daging ayam (dibersihkan dan dipotong dadar)
- 100 gram kacang merah (dihaluskan atau direbus hingga empuk)
- 100 ml air
- 1 sudag rasa (untuk penyesuaian rasanya)
- Garam secukupnya
- Daun bawang dan daun kemangi (untuk tambahan)

#### Langkah-langkah:
1. **Cuci Bersih**: Cuci bersih daging ayam dan kacang merah.

2. **Tumis**: Panaskan minyak dalam wajan, tumis daging ayam hingga setengah matang. Angkat dan sisihkan.

3. **Tumis Bumbu**: Tumis bumbu halus seperti ketumbar, jinten, kunyit, dan serai selama beberapa saat.

4. **Masak Air**: Masukkan kacang merah 

In [49]:
import gradio as gr

def chat_fn(message, history):
    results = retrieve(message, top_k=5)
    answer = generate_answer(message, results)
    sources = ', '.join(sorted(set(f"{r['section_title']} (p.{r['pdf_page']})" for r in results)))
    return f"{answer}\n\n---\n*Sumber: {sources}*"

demo = gr.ChatInterface(
    fn=chat_fn,
    title='RAG Buku KIA — Tanya Seputar Kehamilan, Persalinan & Menyusui (100% Gratis)',
    description='Jawaban berdasarkan Buku KIA (Kemenkes RI), pakai model open-source lokal. Bukan pengganti konsultasi medis.',
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7c92dc38ea65459aa4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [50]:
EXAMPLE_QUESTIONS = [
    'Apa tanda bahaya pada trimester 1?',
    'Bagaimana cara memerah dan menyimpan ASI?',
    'Berapa porsi makan ibu menyusui per hari?',
    'Anak saya batuk, bagaimana penanganannya?',
    'Aktivitas fisik apa yang tidak boleh dilakukan ibu hamil?',
]

In [54]:
import gradio as gr

def format_sources(retrieved_chunks):
    seen = []
    for c in retrieved_chunks:
        label = f"{c['section_title']} · hal. {c['pdf_page']}"
        if label not in seen:
            seen.append(label)
    return seen


def chat_fn(message, history):
    retrieved = retrieve(message, top_k=5)
    result = generate_answer(message, retrieved)

    # Robust terhadap generate_answer yang return string SAJA, atau tuple (answer, has_danger)
    if isinstance(result, tuple):
        answer, has_danger_from_fn = result
        has_danger = has_danger_from_fn
    else:
        answer = result
        has_danger = any(c["is_danger_sign"] for c in retrieved)

    sources = format_sources(retrieved)

    parts = [answer.strip()]

    if has_danger:
        parts.append(
            "\n\n> ⚠️ **Kalau ini kondisi darurat, segera hubungi bidan/dokter "
            "atau ke fasilitas kesehatan terdekat.**"
        )

    if sources:
        sources_text = "  \n".join(f"📄 *{s}*" for s in sources)
        parts.append(f"\n\n---\n**Sumber:**  \n{sources_text}")

    return "".join(parts)

In [52]:
theme = gr.themes.Soft(
    primary_hue=gr.themes.colors.pink,
    secondary_hue=gr.themes.colors.emerald,
    neutral_hue=gr.themes.colors.slate,
    font=[gr.themes.GoogleFont('Poppins'), 'sans-serif'],
).set(
    button_primary_background_fill='*primary_500',
    button_primary_background_fill_hover='*primary_600',
    block_radius='16px',
    block_shadow='0 2px 12px rgba(0,0,0,0.06)',
)

CUSTOM_CSS = """
#header-box {
    background: linear-gradient(135deg, #fbcfe8 0%, #a7f3d0 100%);
    border-radius: 20px;
    padding: 24px 28px;
    margin-bottom: 12px;
}
#header-box h1 { margin: 0 0 4px 0; font-size: 26px; color: #831843; }
#header-box p { margin: 0; color: #065f46; font-size: 14px; }
#disclaimer { font-size: 12px; color: #64748b; text-align: center; margin-top: 8px; }
"""

In [55]:
with gr.Blocks(theme=theme, css=CUSTOM_CSS, title='RAG Buku KIA') as demo:
    gr.HTML("""
        <div id="header-box">
            <h1>🤱 Tanya Buku KIA</h1>
            <p>Asisten seputar kehamilan, persalinan, dan menyusui — berdasarkan
            Buku Kesehatan Ibu dan Anak (Kemenkes RI). Bukan pengganti konsultasi medis.</p>
        </div>
    """)

    gr.ChatInterface(
        fn=chat_fn,
        examples=EXAMPLE_QUESTIONS,
        chatbot=gr.Chatbot(height=480, avatar_images=(None, '🤱')),
        textbox=gr.Textbox(
            placeholder='Tulis pertanyaan seputar kehamilan, persalinan, atau menyusui...'
        ),
    )

    gr.HTML(
        '<div id="disclaimer">⚕️ Informasi ini bersifat umum dan tidak menggantikan '
        'pemeriksaan langsung oleh tenaga kesehatan.</div>'
    )

demo.launch(share=True)

/tmp/ipykernel_7719/155089916.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=theme, css=CUSTOM_CSS, title='RAG Buku KIA') as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9e52eef5c9119aff48.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
